<a href="https://colab.research.google.com/github/betulbilhan2/LLM-Augmentation-Fidelity/blob/main/NB03_LLM_Paraphrase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: LLM Paraphrase Generation

**Goal:**
1. Generate the **LLM Paraphrasing (E3)** group using the Groq API (e.g., Llama 3) for a 1:1 synthetic data generation.
2. Apply this to all 10 seeds across both `low` and `normal` resource levels.
3. Save metadata of the generation (model, parameters, time) to `llm_metadata.json`.


In [ ]:
# Install necessary libraries
!pip install groq pandas pyyaml tenacity

import os
import sys
import yaml
import json
import pandas as pd
import time
from datetime import datetime
from groq import Groq
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.colab import drive
from google.colab import userdata
import logging

# Mount Drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/tr_augmentation_project'

# Setup logging
log_file_path = os.path.join(PROJECT_ROOT, "logs", "nb03_run.log")
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file_path, encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ],
    force=True
)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Load Experiment Config
config_path = os.path.join(PROJECT_ROOT, "configs", "experiment_config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

logger.info("Configuration loaded.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00
Mounted at /content/drive
2026-09-05 15:58:26,772 - INFO - Configuration loaded.


In [ ]:
# Setup Groq Client and Parameters
groq_api_key = userdata.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets. Please add it.")

client = Groq(api_key=groq_api_key)

# Get LLM parameters from config
llm_config = config.get("llm_augmentation", {})
temperature = llm_config.get("temperature", 0)
top_p = llm_config.get("top_p", 1.0)
max_tokens = 500  # reasoning + content için yeterli pay

# Model sabitlendi — dinamik seçim kaldırıldı
model_name = "openai/gpt-oss-120b"
logger.info(f"Using LLM Provider: Groq, Model: {model_name}")

# Define the Prompt
prompt_template = """Sen bir Türkçe metin parafraz aracısın. Aşağıdaki müşteri yorumunu, anlamını ve orijinal duygu tonunu koruyarak farklı kelimeler ve/veya cümle yapılarıyla yeniden yaz.

Kurallar:
* Metnin anlamını ve sentiment sınıfını kesinlikle koru.
* Duyguyu güçlendirme, zayıflatma veya tersine çevirme.
* Orijinal metinde bulunmayan yeni bilgi, olay, görüş veya duygu ekleme.
* Orijinal metindeki önemli bilgileri, ayrıntıları veya anlam taşıyan unsurları çıkarma.
* Kişi, ürün, marka, yer, sayı, fiyat, tarih ve benzeri özel bilgileri değiştirme.
* Yalnızca parafraz yap; özetleme, genişletme veya yorumlama yapma.
* Metni doğal ve akıcı Türkçeyle yeniden yaz.
* Orijinal metin zaten doğal ve anlaşılırsa gereksiz değişiklik yapma.
* SADECE yeniden yazılmış metni döndür.
* Açıklama, gerekçe, selamlama, etiket veya tırnak işareti ekleme.

Metin: {text}
Etiket: {label}"""

# Save prompt to file as required by pipeline
prompts_dir = os.path.join(PROJECT_ROOT, "prompts")
os.makedirs(prompts_dir, exist_ok=True)
prompt_file_path = os.path.join(prompts_dir, "paraphrase_v1.txt")
with open(prompt_file_path, "w", encoding="utf-8") as f:
    f.write(prompt_template)

logger.info(f"Prompt saved to {prompt_file_path}")


2026-09-05 19:08:18,805 - INFO - Using LLM Provider: Groq, Model: openai/gpt-oss-120b
2026-09-05 19:08:18,815 - INFO - Prompt saved to /content/drive/MyDrive/tr_augmentation_project/prompts/paraphrase_v1.txt


In [ ]:
import groq

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60),
    stop=stop_after_attempt(10),
    retry=retry_if_exception_type(groq.RateLimitError)
          | retry_if_exception_type(groq.APIConnectionError)
          | retry_if_exception_type(groq.InternalServerError)
          | retry_if_exception_type(ValueError),
    before_sleep=lambda retry_state: logger.warning(
        f"RETRY SEBEBI: {retry_state.outcome.exception()}"
    )
)
def generate_paraphrase(text, label):
    prompt = prompt_template.format(text=text, label=label)

    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        reasoning_effort="low",
    )
    result = chat_completion.choices[0].message.content.strip()
    if not result:
        raise ValueError(f"Empty content returned for text: {text[:50]}...")
    return result

# Test
test_text = "Ürün elime çok geç ulaştı, hiç memnun kalmadım."
test_label = "negative"
logger.info("Testing Groq API connection...")
test_res = generate_paraphrase(test_text, test_label)
logger.info(f"Test Original: {test_text}")
logger.info(f"Test Paraphrase: {test_res}")

2026-09-05 19:08:18,826 - INFO - Testing Groq API connection...
2026-09-05 19:08:19,279 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-05 19:08:19,283 - INFO - Test Original: Ürün elime çok geç ulaştı, hiç memnun kalmadım.
2026-09-05 19:08:19,284 - INFO - Test Paraphrase: Ürün çok geç elime geldi, hiç memnun kalmadım.


In [ ]:
# Execute Generation Pipeline

DATA_SEEDS = config["seeds"]["data_seeds"]
LEVELS = ["low", "normal"]

total_requests = 0

for level in LEVELS:
    logger.info(f"--- Processing Resource Level: {level.upper()} ---")

    for seed in DATA_SEEDS:
        llm_dir = os.path.join(PROJECT_ROOT, f"02_augmented/{level}/llm_paraphrase/seed_{seed}")
        os.makedirs(llm_dir, exist_ok=True)
        out_path = os.path.join(llm_dir, "pool.csv")

        # Check if already processed (checkpointing)
        if os.path.exists(out_path):
            logger.info(f"  Seed {seed}/9 already processed for {level}. Skipping.")
            continue

        logger.info(f"  Seed {seed}/9 processing...")

        # Load Original Seed Data
        train_path = os.path.join(PROJECT_ROOT, f"01_splits/{level}/seed_{seed}/train_seed.csv")
        df_train = pd.read_csv(train_path)

        paraphrased_texts = []
        for idx, row in df_train.iterrows():
            # Add a small delay to avoid hitting rate limits too aggressively even with retry
            time.sleep(3)

            try:
                res = generate_paraphrase(row['text'], row['label'])
                paraphrased_texts.append(res)
                total_requests += 1

                # Print progress every 10 samples
                if (idx + 1) % 10 == 0:
                    logger.info(f"    Progress: {idx+1}/{len(df_train)} completed.")

            except Exception as e:
                logger.error(f"Failed to generate paraphrase for idx {idx} after retries: {e}")
                # Fallback to original text if API completely fails
                paraphrased_texts.append(row['text'])

        df_llm = df_train.copy()
        df_llm['text'] = paraphrased_texts

        df_llm.to_csv(out_path, index=False)
        logger.info(f"  Saved outputs to {out_path}")

logger.info(f"✅ LLM Paraphrasing completed. Total API requests made: {total_requests}")


2026-09-05 19:08:19,298 - INFO - --- Processing Resource Level: LOW ---
2026-09-05 19:08:19,303 - INFO -   Seed 0/9 already processed for low. Skipping.
2026-09-05 19:08:19,308 - INFO -   Seed 1/9 already processed for low. Skipping.
2026-09-05 19:08:19,312 - INFO -   Seed 2/9 already processed for low. Skipping.
2026-09-05 19:08:19,316 - INFO -   Seed 3/9 already processed for low. Skipping.
2026-09-05 19:08:19,320 - INFO -   Seed 4/9 already processed for low. Skipping.
2026-09-05 19:08:19,324 - INFO -   Seed 5/9 already processed for low. Skipping.
2026-09-05 19:08:19,328 - INFO -   Seed 6/9 already processed for low. Skipping.
2026-09-05 19:08:19,332 - INFO -   Seed 7/9 already processed for low. Skipping.
2026-09-05 19:08:19,336 - INFO -   Seed 8/9 already processed for low. Skipping.
2026-09-05 19:08:19,340 - INFO -   Seed 9/9 already processed for low. Skipping.
2026-09-05 19:08:19,341 - INFO - --- Processing Resource Level: NORMAL ---
2026-09-05 19:08:19,345 - INFO -   Seed 0/9

In [ ]:
# Save metadata
metadata = {
    "provider": "groq",
    "exact_model_id": model_name,
    "temperature": temperature,
    "top_p": top_p,
    "max_tokens": max_tokens,
    "prompt_file": "prompts/paraphrase_v1.txt",
    "generation_date": datetime.now().isoformat()
}

meta_path = os.path.join(PROJECT_ROOT, "configs", "llm_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

logger.info(f"Metadata saved to {meta_path}")


2026-09-05 19:43:12,362 - INFO - Metadata saved to /content/drive/MyDrive/tr_augmentation_project/configs/llm_metadata.json


In [ ]:
# Force flush to Drive to prevent data loss or 0 byte files
import logging
for handler in logging.root.handlers:
    handler.flush()
if 'logger' in globals():
    for handler in logger.handlers:
        handler.flush()

from google.colab import drive
drive.flush_and_unmount()
print("Drive'a yazma işlemi başarıyla tamamlandı ve bağlantı sonlandırıldı. Drive'ı kontrol edebilirsiniz.")


Drive'a yazma işlemi başarıyla tamamlandı ve bağlantı sonlandırıldı. Drive'ı kontrol edebilirsiniz.
